# LỊCH SỬ SUPERVISED · deeplabv3plus_resnet50

Không dùng notebook này cho DG. Campaign đã bị chặn mặc định; ACDC có nhãn từng được dùng để học.

Một model, hai giai đoạn: **Cityscapes 20 epoch → ACDC replay 20 epoch**. Recipe hiện có, không phải notebook baseline checkpoint công bố mới. Encoder ImageNet + head mới;không nạp checkpoint project cũ. B0 đã ngừng sử dụng.

**Fresh:** code ZIP mới + Cityscapes + ACDC, GPU + Internet. **Resume:** ZIP của chính model này + cùng dataset/mount;điền `RESUME_INPUT`. Khi resume,source code được lấy từ snapshot của lượt chạy đó.

Mỗi phiên dự báo tối đa10giờ;một model có thể cần nhiều phiên. Sau mỗi epoch:transactional last/best,previous generation và atomic resultZIP. Epoch dang dở sẽ chạy lại. Dùng SaveVersion→Save&RunAll và giữ ZIP/output giữa các phiên;checkpoint local không tự đồng bộ ra Drive.

## 1. Input và khôi phục


In [ ]:
"""Standard-library source bootstrap shared by the three model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_research_campaign.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','requirements-demo.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os,sys,json,hashlib,time
NOTEBOOK_STARTED=time.monotonic()
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
import torch
assert torch.cuda.is_available(),'Settings -> Accelerator -> GPU'
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__)

MODEL_NAME='deeplabv3plus_resnet50'
CODE_INPUT=Path('/kaggle/input')
CITYSCAPES_ROOT=None
ACDC_ROOT=None
RESUME_INPUT=None  # This model's result ZIP or extracted dataset root
SEED=42
WORK=Path('/kaggle/working/campaign_deeplabv3plus_resnet50')
REPO=Path('/kaggle/working/project_deeplabv3plus_resnet50')
ARCHIVE=Path('/kaggle/working/deeplabv3plus_resnet50_seed42.zip')
assert not WORK.exists() and not REPO.exists(),'Use a fresh session;preserve the previous result ZIP first'
if RESUME_INPUT is None:
    copy_source(resolve_source(CODE_INPUT,Path('/kaggle/input')),REPO)
    WORK.mkdir(parents=True)
else:
    previous=Path(RESUME_INPUT)
    if previous.is_file():extract_safe(previous,WORK)
    else:
        found=sorted({p.parent for p in previous.rglob('campaign.json') if (p.parent/'data/data.json').is_file()})
        if (previous/'campaign.json').is_file():found=[previous]
        assert len(found)==1,f'Set RESUME_INPUT to this model workspace: {found}'
        shutil.copytree(found[0],WORK)
    plan=json.loads((WORK/'campaign.json').read_text())
    assert plan['schema']=='selected_models_fresh_ce_20e_v1','Use this model notebook result;do not mix the old multi-model campaign'
    assert [j['model'] for j in plan['jobs']]==[MODEL_NAME,MODEL_NAME],'Result belongs to a different model'
    source=WORK/'source.zip' if (WORK/'source.zip').is_file() else WORK/'source'
    copy_source(source,REPO)
    assert all((REPO/name).is_file() and hashlib.sha256((REPO/name).read_bytes()).hexdigest()==checksum
               for name,checksum in plan['source_hashes'].items()),'Original source mismatch'
    if not (WORK/'source.zip').is_file():
        with zipfile.ZipFile(WORK/'source.zip','w',zipfile.ZIP_DEFLATED) as z:
            for name in plan['source_hashes']:z.write(REPO/name,name)
            for name in ('requirements.txt','THIRD_PARTY_NOTICES.md','third_party/transformers/LICENSE'):
                if (REPO/name).is_file():z.write(REPO/name,name)
    SEED=plan['seed']

def unique_root(hint,marker,required):
    if hint is not None:root=Path(hint)
    else:
        found=sorted({p.parent for p in Path('/kaggle/input').rglob(marker)
                      if p.is_dir() and all((p.parent/item).is_dir() for item in required)})
        assert len(found)==1,f'Set dataset root explicitly: {found}'
        root=found[0]
    assert all((root/item).is_dir() for item in required),f'Invalid root: {root}'
    return root
CITYSCAPES_ROOT=unique_root(CITYSCAPES_ROOT,'leftImg8bit',('leftImg8bit/train','gtFine/train'))
ACDC_ROOT=unique_root(ACDC_ROOT,'rgb_anon_trainvaltest',('rgb_anon_trainvaltest/rgb_anon','gt_trainval/gt'))
os.chdir(REPO);sys.path.insert(0,str(REPO))
print('Model:',MODEL_NAME,'Data:',CITYSCAPES_ROOT,ACDC_ROOT)


## 2. Môi trường và dữ liệu

Dataset root tự tìm theo cấu trúc. Torch/TorchVision dùng bản sẵn có của Kaggle. Giữ dataset version/mount khi resume.


In [ ]:
import subprocess
def run(*args):subprocess.run([sys.executable,'-u',*map(str,args)],check=True,cwd=REPO)
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow>=10,<12',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torchvision
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,6),'Use Torch>=2.6'
print('TorchVision:',torchvision.__version__)
from src.experiments.packaging import export_workspace
from scripts.run_research_campaign import prepare_campaign
from IPython.display import display,FileLink
try:
    run('-m','scripts.prepare_research_data','--city',CITYSCAPES_ROOT,'--acdc',ACDC_ROOT,'--out',WORK/'data')
    prepare_campaign(WORK,WORK/'data/data.json',SEED,models=[MODEL_NAME])
except Exception:
    export_workspace(WORK,ARCHIVE)
    print('Preserved setup files:',ARCHIVE)
    raise


## 3. Train source rồi tự chuyển sang ACDC

Mỗi epoch cập nhật ZIP. Khi gần hết cửa sổ phiên,queue dừng ở checkpoint;phiên sau dùng `RESUME_INPUT` để tiếp tục. ETA sau epoch đầu dựa trên thời gian thực tế của giai đoạn đang chạy;chưa đo giai đoạn sau thì chưa có ETA chính xác cho cả lượt.


In [ ]:
remaining=10-(time.monotonic()-NOTEBOOK_STARTED)/3600
try:
    if remaining>0:
        run('-m','scripts.run_research_campaign','--root',WORK,'--data',WORK/'data/data.json',
            '--archive',ARCHIVE,'--models',MODEL_NAME,'--seed',SEED,'--session-hours',remaining)
    else:print('Setup used the work window;preserving for a fresh session.')
finally:
    export_workspace(WORK,ARCHIVE)
    print('Download:',ARCHIVE)
    display(FileLink(str(ARCHIVE)))


## 4. Tiến độ và kết quả

`training_complete` nghĩa là đủ hai giai đoạn×20epoch của model này. Điểm selection là monitor/dev;official-val được chấm riêng sau khi khóa checkpoint.


In [ ]:
from IPython.display import Markdown
from PIL import Image
if (WORK/'status.json').is_file():
    report=json.loads((WORK/'status.json').read_text())
    print('Status:',report['status'])
    rows=['| Stage | Epoch | Best | Selection mIoU | Remaining stage ETA |',
          '|---|---:|---:|---:|---:|']
    for job in report.get('jobs',[]):
        score=job.get('selection_miou');score='pending' if score is None else f'{100*score:.2f}%'
        eta=job.get('estimated_remaining_hours');eta='unmeasured' if eta is None else f'{eta:.1f}h + ZIP/setup'
        rows.append(f"| {job['stage']} | {job['completed_epochs']}/20 | {job.get('best_epoch','')} | {score} | {eta} |")
    display(Markdown('\n'.join(rows)))
for path in sorted((WORK/'jobs').glob('*/run/report/learning_curves.png')):
    print(path.parent.parent.parent.name);display(Image.open(path))
